# Semana 7 — CAPM Empírico

**Econolab Financial Economics**

---

## Objetivo
Estimar el modelo CAPM con datos reales y entender lo que beta *realmente* mide.

## El modelo
$$E[r_i] = r_f + \beta_i \cdot (E[r_m] - r_f)$$

Donde $\beta_i$ se estima como la pendiente de:
$$r_i - r_f = \alpha_i + \beta_i (r_m - r_f) + \varepsilon_i$$

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from data.generate_data import generate_stock_prices, generate_market_index
from src.returns import log_returns
from src.capm import estimate_beta, capm_expected_return, security_market_line, rolling_beta, multi_asset_betas

prices  = generate_stock_prices()
market  = generate_market_index(prices)

# Calcular retornos
all_returns = {col: log_returns(prices[col]) for col in prices.columns}
market_ret  = log_returns(market)

risk_free_daily = 0.04 / 252
print('Datos cargados ✓')

## 1. Estimación de Beta para un activo

Beta se estima mediante regresión OLS de los retornos en exceso del activo sobre los retornos en exceso del mercado.

In [ ]:
# Estimamos beta para TECH
tech_ret = all_returns['TECH']
result   = estimate_beta(tech_ret, market_ret)

print('═' * 50)
print('  CAPM — Activo: TECH')
print('═' * 50)
print(f'  Alpha (Jensen):  {result["alpha"]*252*100:.3f}% anual')
print(f'  Beta:            {result["beta"]:.4f}')
print(f'  R²:              {result["r_squared"]:.4f}')
print(f'  Beta p-value:    {result["beta_pvalue"]:.6f}')
print(f'  Observaciones:   {result["n_obs"]}')
print('─' * 50)
print(f'  Interpretación: TECH es {result["beta"]:.1f}x más volátil que el mercado')

# Retorno esperado según CAPM
expected = capm_expected_return(result['beta'])
print(f'  Retorno esperado (CAPM): {expected*100:.2f}% anual')

## 2. Scatter plot: Characteristic Line

In [ ]:
# Retornos en exceso
excess_tech   = tech_ret - risk_free_daily
excess_market = market_ret - risk_free_daily

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(excess_market, excess_tech, alpha=0.35, s=15, color='#2563EB', label='Observaciones')

# Línea de regresión
x_line = np.linspace(excess_market.min(), excess_market.max(), 100)
y_line = result['alpha'] + result['beta'] * x_line
ax.plot(x_line, y_line, color='#DC2626', linewidth=2.5,
        label=f'β = {result["beta"]:.3f}  |  R² = {result["r_squared"]:.3f}')

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Retorno exceso Mercado')
ax.set_ylabel('Retorno exceso TECH')
ax.set_title('Línea Característica — TECH vs Mercado', fontsize=13, fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Comparación de Betas — Todos los activos

In [ ]:
betas_df = multi_asset_betas(prices, market)
betas_df

In [ ]:
# Línea del Mercado de Valores (SML)
betas_range = np.linspace(0, 2, 100)
sml         = security_market_line(betas_range)

fig, ax = plt.subplots(figsize=(9, 6))

# SML
ax.plot(betas_range, sml * 100, color='#475569', linewidth=2, label='SML (CAPM teórico)', zorder=1)

# Activos reales
colors = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED']
for i, (asset_name, row) in enumerate(betas_df.iterrows()):
    actual_ret = log_returns(prices[asset_name]).mean() * 252 * 100
    expected_ret = row['E[r] CAPM (%)']
    beta = row['Beta']
    
    ax.scatter(beta, actual_ret, s=120, color=colors[i], zorder=3, label=asset_name)
    ax.annotate(asset_name, (beta, actual_ret), textcoords='offset points',
                xytext=(8, 4), fontsize=9, color=colors[i])

ax.axhline(0.04 * 100, color='gray', linewidth=1, linestyle=':', alpha=0.7)
ax.text(0.02, 4.3, 'r_f = 4%', fontsize=9, color='gray')

ax.set_xlabel('Beta (β)', fontsize=12)
ax.set_ylabel('Retorno anual (%)', fontsize=12)
ax.set_title('Línea del Mercado de Valores (SML)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Beta Rodante — ¿Es beta constante?

Una crítica real al CAPM: beta **no es constante** en el tiempo.

In [ ]:
roll_b = rolling_beta(all_returns['TECH'], market_ret, window=63)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(roll_b.index, roll_b.values, color='#7C3AED', linewidth=1.8)
ax.axhline(1.0, color='#DC2626', linewidth=1, linestyle='--', label='β = 1 (mercado)')
ax.axhline(roll_b.mean(), color='#2563EB', linewidth=1, linestyle='--',
           label=f'β promedio = {roll_b.mean():.2f}')
ax.fill_between(roll_b.index, roll_b.values, 1.0, alpha=0.15, color='#7C3AED')
ax.set_title('Beta Rodante — TECH (ventana 63 días)', fontsize=13, fontweight='bold')
ax.set_ylabel('Beta')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Beta promedio: {roll_b.mean():.3f}')
print(f'Rango: [{roll_b.min():.3f}, {roll_b.max():.3f}]')

---

## 📝 Ejercicio

1. Un activo con beta = 1.8 ¿qué significa económicamente? Da un ejemplo de qué tipo de empresa podría tener ese beta.
2. Identifica en el gráfico SML cuáles activos están **por encima** y **por debajo** de la línea. ¿Cuáles están subvalorados según el CAPM?
3. ¿Por qué el beta rodante cambia? ¿Qué factores económicos pueden explicar esos cambios?

**Implementa la estimación de beta para BANK y compara con TECH:**

In [ ]:
# TODO: estima beta para BANK y ENERGY
# Compara sus betas e interpreta la diferencia económica
